# 🚀 FaceKey Collector — Google Colab (Clean & Simplified)
### Dedicated Worker: `colab-worker-1` (3 Parallel Workers, Turbo 150+ FPS, Direct 5 TB Drive Storage)
- **Cell 1**: **1-Click Runner**. Starts the multi-worker collector. Every video, it **automatically saves chunks directly into your 5 TB Google Drive** (`FaceKeyDataset/chunks/`). No manual save needed!
- **Cell 2**: **Emergency Save / Flush**. If you ever stop the collector early, click this cell to immediately package any unfinished session into Google Drive.
- **Cell 3**: **Pull Kaggle & Lightning Data (Duplicate-Proof)**. 1-click cloud-to-cloud delta sync that brings ONLY new chunks into your 5 TB Google Drive without any duplicate files or re-downloads!

## ⚡ CELL 1: 1-CLICK RUNNER (CONTINUOUS AUTO-SAVE TO 5 TB GOOGLE DRIVE)

In [ ]:
# ==============================================================================
# ⚡ CELL 1: 1-CLICK ALL-IN-ONE RUNNER (DIRECT GOOGLE DRIVE AUTO-SAVE)
# ==============================================================================
import os, sys, shutil

# 1. Mount 5 TB Google Drive
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    print("[*] Connecting to Google Drive...")
    drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/FaceKeyDataset"
for folder in ("chunks", "checkpoints", "locks", "data"):
    os.makedirs(f"{DRIVE_DIR}/{folder}", exist_ok=True)
print(f"[+] 5 TB Google Drive ready at: {DRIVE_DIR}")

# 2. Update to latest clean repository code (Self-Healing)
%cd /content
APP_DIR = "/content/FaceKeyAnimation"
if os.path.isdir(os.path.join(APP_DIR, ".git")):
    print("[*] Updating FaceKeyAnimation to latest GitHub main...")
    !cd {APP_DIR} && git fetch --all --prune && git reset --hard origin/main && git clean -fd
else:
    print("[*] Fresh clone of FaceKeyAnimation from GitHub...")
    !rm -rf {APP_DIR}
    !git clone https://github.com/Vijay-1010110/FaceKeyAnimation.git {APP_DIR}

%cd {APP_DIR}

# 3. Ensure Cloud Dependencies & Deno JS solver are present
!pip install -q -U --pre "yt-dlp[default]" mediapipe opencv-python-headless numpy sounddevice mss av scipy huggingface_hub pysocks
!if ! command -v deno &> /dev/null; then curl -fsSL https://deno.land/install.sh | sh > /dev/null 2>&1 && ln -sf /root/.deno/bin/deno /usr/local/bin/deno; fi

# 4. Sync YouTube Queue
urls_file = f"{DRIVE_DIR}/youtube_urls_colab.txt"
repo_colab = os.path.join(APP_DIR, "sessions", "youtube_urls_colab.txt")
if not os.path.exists(urls_file) or os.path.getsize(urls_file) == 0:
    if os.path.exists(repo_colab):
        shutil.copy2(repo_colab, urls_file)

# 5. Continuous Cloud Backup (Hugging Face write token)
HF_TOKEN = os.environ.get('HF_TOKEN', '')
token_file = f"{DRIVE_DIR}/hf_token.txt"
if not HF_TOKEN and os.path.exists(token_file):
    with open(token_file) as f:
        HF_TOKEN = f.read().strip()
HF_REPO = 'VijayTheOne/facekey-dataset-chunks'

# 6. Launch Collector (3 Workers, 1 Face Only, Turbo Scratch Mode, Saves every single video)
WORKER_ID = "colab-worker-1"
print(f"\n[*] Launching FaceKey Turbo Cloud Collector as '{WORKER_ID}'...")
!python scripts/cloud_data_collector.py \
    --drive-dir "{DRIVE_DIR}" \
    --worker-id "{WORKER_ID}" \
    --urls-file "{urls_file}" \
    --chunk-size 1 \
    --quality 480p \
    --num-workers 3 \
    --hf-token '{HF_TOKEN}' \
    --hf-repo '{HF_REPO}'


---
## 🚨 CELL 2: EMERGENCY SAVE & FLUSH TO GOOGLE DRIVE
> [!TIP]
> Run this cell if you stop the collector early. It immediately packages any partial sessions into Google Drive (`FaceKeyDataset/chunks/`).

In [ ]:
# ==============================================================================
# 🚨 CELL 2: FLUSH & SAVE ALL UNFINISHED SESSIONS DIRECTLY TO GOOGLE DRIVE
# ==============================================================================
%cd /content/FaceKeyAnimation
from scripts.cloud_sync import CloudSyncManager
cs = CloudSyncManager('/content/drive/MyDrive/FaceKeyDataset')
created = cs.pack_sessions_into_chunk('/content/FaceKeyAnimation/sessions', 'colab-worker-1')
if created:
    print(f'[SUCCESS] Created and saved emergency chunk directly to Google Drive: {created}')
else:
    print('[+] All sessions are already packaged and saved in Google Drive!')


---
## 📥 CELL 3: PULL ONLY NEW KAGGLE & LIGHTNING DATA INTO 5 TB DRIVE
> [!TIP]
> **100% Duplicate-Proof**: Checks what is already in `/content/drive/MyDrive/FaceKeyDataset/chunks/`. Skips existing files (0 bytes downloaded), and pulls **ONLY brand new chunks** from Hugging Face!

In [ ]:
# ==============================================================================
# 📥 CELL 3: SMART DUPLICATE-PROOF DELTA SYNC INTO 5 TB GOOGLE DRIVE
# ==============================================================================
import os

# 1. Update repo to ensure latest duplicate-proof sync script
%cd /content/FaceKeyAnimation
!git pull origin main

# 2. Run smart delta-sync (SKIPS all existing Drive chunks, downloads ONLY new chunks)
!python scripts/sync_hf_to_drive.py --drive-dir '/content/drive/MyDrive/FaceKeyDataset'
